1. Read image file paths from the normal / scratch / spot folders
2. Randomly sample a few images from a chosen class
3. Load each image as grayscale PIL image
4. Apply a chosen enhancement:
      scratch -> sharpen + contrast
      spot -> autocontrast + contrast
5. Create a side-by-side comparison image
6. Display the contact sheet directly using PIL

In [ ]:
import os
import random
import numpy as np
from PIL import Image, ImageFilter, ImageEnhance, ImageOps
from IPython.display import display

BASE_DIR = "<REPO_ROOT>/Implementation_trial/cDCGAN_SDI/data/prepared_A"   # <-- update this
NORMAL_DIR = os.path.join(BASE_DIR, "normal")
SCRATCH_DIR = os.path.join(BASE_DIR, "scratches")
SPOT_DIR = os.path.join(BASE_DIR, "spots")

IMG_SIZE = 128
SEED = 42
random.seed(SEED)

def list_images(folder):
    return sorted([
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
    ])

normal_files = list_images(NORMAL_DIR)
scratch_files = list_images(SCRATCH_DIR)
spot_files = list_images(SPOT_DIR)

def load_gray_pil(path, size=IMG_SIZE):
    img = Image.open(path).convert("L")
    img = img.resize((size, size))
    return img

"""
Scratches are:

thin
elongated
often edge-like

So sharpening + stronger contrast is a reasonable first attempt.
"""
def enhance_scratch(img_pil):
    enhanced = img_pil.filter(ImageFilter.SHARPEN) # applies a standard sharpen filter. this boosts local edge-like details
    enhanced = ImageEnhance.Contrast(enhanced).enhance(1.9) # increases contrast by a factor of 1.9. makes bright/dark differences stronger
    enhanced = enhanced.filter(ImageFilter.SHARPEN) #applies another sharpening pass. this is a simple way to intensify edge emphasis
    return enhanced # returns the transformed PIL image

"""
Spots are more:

localized
intensity-based
blob-like

So contrast-based enhancement is usually more appropriate than aggressive sharpening.
"""
def enhance_spot(img_pil):
    enhanced = ImageOps.autocontrast(img_pil) # stretches the image intensity range automatically. this can make hidden local differences more visible
    enhanced = ImageEnhance.Contrast(enhanced).enhance(1.9) # further increases contrast
    return enhanced # returns the transformed PIL image

def make_contact_sheet(pil_images, titles=None, cols=2, thumb_size=(128, 128), padding=10, title_height=20):
    rows = (len(pil_images) + cols - 1) // cols
    sheet_w = cols * thumb_size[0] + (cols + 1) * padding
    sheet_h = rows * (thumb_size[1] + title_height) + (rows + 1) * padding
    sheet = Image.new("L", (sheet_w, sheet_h), color=255)

    for idx, img in enumerate(pil_images):
        r = idx // cols
        c = idx % cols
        x = padding + c * thumb_size[0]
        y = padding + r * (thumb_size[1] + title_height)
        sheet.paste(img.resize(thumb_size), (x, y + title_height))

    return sheet

def preview_pairs(file_list, transform_fn, n=5):
    chosen = random.sample(file_list, min(n, len(file_list)))
    images = []

    for path in chosen:
        original = load_gray_pil(path)
        enhanced = transform_fn(original)
        images.extend([original, enhanced])

    sheet = make_contact_sheet(images, cols=2, thumb_size=(128, 128))
    display(sheet)

def preview_normals_both(n=5):
    chosen = random.sample(normal_files, min(n, len(normal_files)))
    images = []

    for path in chosen:
        original = load_gray_pil(path)
        scratch_like = enhance_scratch(original)
        spot_like = enhance_spot(original)
        images.extend([original, scratch_like, spot_like])

    sheet = make_contact_sheet(images, cols=3, thumb_size=(128, 128))
    display(sheet)

print("Scratch preview:")
preview_pairs(scratch_files, enhance_scratch, n=5)

print("Spot preview:")
preview_pairs(spot_files, enhance_spot, n=5)

print("Normals under both transforms:")
preview_normals_both(n=5)

In [ ]:
import os
import random
import numpy as np
from PIL import Image, ImageFilter, ImageEnhance, ImageOps
from IPython.display import display

BASE_DIR = "<REPO_ROOT>/Implementation_trial/cDCGAN_SDI/data/prepared_A"   # <-- update this
NORMAL_DIR = os.path.join(BASE_DIR, "normal")
SCRATCH_DIR = os.path.join(BASE_DIR, "scratches")
SPOT_DIR = os.path.join(BASE_DIR, "spots")

IMG_SIZE = 128
SEED = 42
random.seed(SEED)

def list_images(folder):
    return sorted([
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
    ])

normal_files = list_images(NORMAL_DIR)
scratch_files = list_images(SCRATCH_DIR)
spot_files = list_images(SPOT_DIR)

def load_gray_pil(path, size=IMG_SIZE):
    img = Image.open(path).convert("L")
    img = img.resize((size, size))
    return img

def enhance_scratch(img_pil):
    enhanced = img_pil.filter(ImageFilter.SHARPEN) # applies a standard sharpen filter. this boosts local edge-like details
    enhanced = ImageEnhance.Contrast(enhanced).enhance(1.9) # increases contrast by a factor of 1.9. makes bright/dark differences stronger
    enhanced = enhanced.filter(ImageFilter.SHARPEN) #applies another sharpening pass. this is a simple way to intensify edge emphasis
    return enhanced

def enhance_spot(img_pil):
    enhanced = ImageOps.autocontrast(img_pil)
    enhanced = ImageEnhance.Contrast(enhanced).enhance(1.9)
    return enhanced

def make_contact_sheet(pil_images, titles=None, cols=2, thumb_size=(128, 128), padding=10, title_height=20):
    rows = (len(pil_images) + cols - 1) // cols
    sheet_w = cols * thumb_size[0] + (cols + 1) * padding
    sheet_h = rows * (thumb_size[1] + title_height) + (rows + 1) * padding
    sheet = Image.new("L", (sheet_w, sheet_h), color=255)

    for idx, img in enumerate(pil_images):
        r = idx // cols
        c = idx % cols
        x = padding + c * thumb_size[0]
        y = padding + r * (thumb_size[1] + title_height)
        sheet.paste(img.resize(thumb_size), (x, y + title_height))

    return sheet

def preview_pairs(file_list, transform_fn, n=5):
    chosen = random.sample(file_list, min(n, len(file_list)))
    images = []

    for path in chosen:
        original = load_gray_pil(path)
        enhanced = transform_fn(original)
        images.extend([original, enhanced])

    sheet = make_contact_sheet(images, cols=2, thumb_size=(128, 128))
    display(sheet)

def preview_normals_both(n=5):
    chosen = random.sample(normal_files, min(n, len(normal_files)))
    images = []

    for path in chosen:
        original = load_gray_pil(path)
        scratch_like = enhance_scratch(original)
        spot_like = enhance_spot(original)
        images.extend([original, scratch_like, spot_like])

    sheet = make_contact_sheet(images, cols=3, thumb_size=(128, 128))
    display(sheet)

print("Scratch preview:")
preview_pairs(scratch_files, enhance_scratch, n=5)

print("Spot preview:")
preview_pairs(spot_files, enhance_spot, n=5)

print("Normals under both transforms:")
preview_normals_both(n=5)

In [ ]:
import matplotlib.pyplot as plt

d_losses = np.load("<REPO_ROOT>/Implementation_trial/cDCGAN_SDI/scripts/batch16_v2_normalVscratch/exp4_normal_vs_scratch/losses/d_losses.npy")
g_losses = np.load("<REPO_ROOT>/Implementation_trial/cDCGAN_SDI/scripts/batch16_v2_normalVscratch/exp4_normal_vs_scratch/losses/g_losses.npy")

def save_loss_plot(d_losses, g_losses):
    plt.figure(figsize=(10, 5))
    plt.plot(d_losses, label="D loss", marker='o', markersize=2, alpha=0.7)
    plt.plot(g_losses, label="G loss", marker='s', markersize=2, alpha=0.7)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("cDCGAN Training Losses")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    loss_plot_path = os.path.join("<REPO_ROOT>/Implementation_trial/cDCGAN_SDI/scripts/batch16_v2_normalVscratch/exp4_normal_vs_scratch/losses", "loss_plot.png")
    plt.savefig(loss_plot_path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Saved loss plot to: {loss_plot_path}")

def main():
    save_loss_plot(d_losses, g_losses)


if __name__ == "__main__":
    main()